# DCG Multiview Refactor - Colab T4 Training Notebook

This notebook is prepared for Google Colab with a T4 GPU.

## What it does
1. Verifies GPU and CUDA.
2. Clones your repository (or uses Google Drive copy).
3. Installs required Python packages.
4. Checks shared dataset files under `datasets/`.
5. Runs training with configurable dataset and epochs.

Set Runtime -> Change runtime type -> GPU before running.

In [ ]:
import torch
import subprocess

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'])

In [ ]:
# Optional: mount Google Drive if your repo/data are stored there
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# # Clone repository or point to an existing project folder
# # Option A: clone from GitHub
# REPO_URL = 'https://github.com/EddarirOmar/Diffusion-based-approaches-for-IMVC.git'
# PROJECT_ROOT = '/content/Diffusion-based-approaches'

# # Option B: if using Drive, set PROJECT_ROOT manually and skip clone
# # PROJECT_ROOT = '/content/drive/MyDrive/path-to-your-repo/Diffusion-based approaches'

# import os
# import subprocess

# if not os.path.exists(PROJECT_ROOT):
#     subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)

# DCG_DIR = os.path.join(PROJECT_ROOT, 'DCG (NEW)')
# print('DCG_DIR:', DCG_DIR)
# print('Exists:', os.path.exists(DCG_DIR))

In [ ]:
import os
import subprocess

BASE_DIR = "/content"

# OPTION A: Download from GitHub (ZIP)
USE_GITHUB = True

if USE_GITHUB:
    PROJECT_ROOT = os.path.join(BASE_DIR, "Diffusion-based-approaches")
    ZIP_PATH = os.path.join(BASE_DIR, "repo.zip")
    EXTRACTED_FOLDER = os.path.join(BASE_DIR, "Diffusion-based-approaches-for-IMVC-main")
    DCG_SRC = os.path.join(EXTRACTED_FOLDER, "DCG (NEW)")
    DATASETS_SRC = os.path.join(EXTRACTED_FOLDER, "datasets")
    DCG_DIR = os.path.join(PROJECT_ROOT, "DCG (NEW)")
    DATASETS_DIR = os.path.join(PROJECT_ROOT, "datasets")

    os.makedirs(PROJECT_ROOT, exist_ok=True)

    # Download ZIP
    subprocess.run([
        "wget", "-q", "-O", ZIP_PATH,
        "https://github.com/EddarirOmar/Diffusion-based-approaches-for-IMVC/archive/refs/heads/main.zip"
    ], check=True)

    # Extract DCG (NEW) and datasets folders
    subprocess.run([
        "unzip", "-q", ZIP_PATH,
        "Diffusion-based-approaches-for-IMVC-main/DCG (NEW)/*",
        "Diffusion-based-approaches-for-IMVC-main/datasets/*"
    ], check=True)

    # Refresh destination folders
    if os.path.exists(DCG_DIR):
        subprocess.run(["rm", "-rf", DCG_DIR], check=True)
    if os.path.exists(DATASETS_DIR):
        subprocess.run(["rm", "-rf", DATASETS_DIR], check=True)

    subprocess.run(["mv", DCG_SRC, PROJECT_ROOT], check=True)
    subprocess.run(["mv", DATASETS_SRC, PROJECT_ROOT], check=True)

# OPTION B: Use Google Drive
else:
    PROJECT_ROOT = "/content/drive/MyDrive/path-to-your-repo/Diffusion-based-approaches"
    DCG_DIR = os.path.join(PROJECT_ROOT, "DCG (NEW)")
    DATASETS_DIR = os.path.join(PROJECT_ROOT, "datasets")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DCG_DIR:", DCG_DIR)
print("DATASETS_DIR:", DATASETS_DIR)
print("DCG exists:", os.path.exists(DCG_DIR))
print("datasets exists:", os.path.exists(DATASETS_DIR))

In [ ]:
# Install required dependencies for this project
%pip install -q numpy scipy scikit-learn munkres

In [ ]:
# Quick dataset presence check (shared datasets folder)
import os

data_dir = DATASETS_DIR
required = [
    'synthetic3d.mat',
    'cub_googlenet_doc2vec_c10.mat',
]

print('Data dir:', data_dir)
for f in required:
    p = os.path.join(data_dir, f)
    print(f, '->', 'FOUND' if os.path.exists(p) else 'MISSING')

In [ ]:
# Optional quick sanity check
%cd {DCG_DIR}
!python smoke_test.py --dataset synthetic3d

## Lambda Tuning (Optuna)

Run this section to search for the best `lambda_*` values before your full training run.

Notes:
- Start with `Synthetic3d` for quick iteration.
- Increase `N_TRIALS` and `TUNE_EPOCHS` for stronger results.
- The objective is mean of `(ACC + NMI + ARI) / 3`.

In [ ]:
%pip install -q optuna

In [ ]:
import os
import json
import random
import itertools
from datetime import datetime

import numpy as np
import torch
import optuna
import pandas as pd

from configure import get_default_config
from datasets import load_data
from get_indicator_matrix_A import get_mask
from ICDM import icdm

# Tuning controls
TUNE_DATASET = 'Synthetic3d'   # e.g. Synthetic3d, CUB, HandWritten, Multi-Fashion, LandUse_21
TUNE_MISSING_RATE = 0.3
TUNE_EPOCHS = 30
N_TRIALS = 20
SEED = 42

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Tuning device:', DEVICE)


def build_optimizer(model, lr):
    ae_params = itertools.chain.from_iterable(ae.parameters() for ae in model.autoencoders)
    df_params = itertools.chain.from_iterable(df.parameters() for df in model.dfs)
    return torch.optim.Adam(
        itertools.chain(ae_params, df_params, model.clusterLayer.parameters(), model.AttentionLayer.parameters()),
        lr=lr,
    )


def prepare_inputs(x_list, missing_rate, device):
    n_views = len(x_list)
    n_samples = x_list[0].shape[0]
    mask = get_mask(n_views, n_samples, missing_rate)

    x_train_list = []
    for v in range(n_views):
        x_masked = x_list[v] * mask[:, v][:, np.newaxis]
        x_train_list.append(torch.from_numpy(x_masked).float().to(device))

    return x_train_list, torch.from_numpy(mask).long().to(device)


def objective(trial):
    config = get_default_config(TUNE_DATASET)
    config['dataset'] = TUNE_DATASET
    config['training']['epoch'] = TUNE_EPOCHS
    config['print_num'] = max(1, TUNE_EPOCHS)
    config['training']['missing_rate'] = TUNE_MISSING_RATE

    # Search space for lambda_* and MMI calibration knobs
    config['training']['lambda_rec'] = trial.suggest_float('lambda_rec', 1e-2, 1e1, log=True)
    config['training']['lambda_df'] = trial.suggest_float('lambda_df', 1e-2, 1e1, log=True)
    config['training']['lambda_ce'] = trial.suggest_float('lambda_ce', 1e-2, 1e1, log=True)
    config['training']['lambda_mmi'] = trial.suggest_float('lambda_mmi', 1e-4, 1.0, log=True)
    config['training']['lambda_cluster'] = trial.suggest_float('lambda_cluster', 1e-3, 1.0, log=True)
    config['training']['lambda_hc'] = trial.suggest_float('lambda_hc', 1e-3, 1.0, log=True)
    config['training']['mmi_temperature'] = trial.suggest_float('mmi_temperature', 0.05, 1.0, log=True)
    config['training']['mmi_internal_lambda'] = trial.suggest_float('mmi_internal_lambda', 0.1, 2.0)

    # Keep seeds deterministic trial-to-trial
    run_seed = SEED + trial.number
    random.seed(run_seed)
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(run_seed)

    x_list, y_list = load_data(config)
    x_train_list, mask = prepare_inputs(x_list, config['training']['missing_rate'], DEVICE)

    model = icdm(config)
    model.to_device(DEVICE)
    optimizer = build_optimizer(model, config['training']['lr'])

    acc, nmi, ari = model.train(config, x_train_list, y_list, mask, optimizer, DEVICE)
    score = float((acc + nmi + ari) / 3.0)

    trial.set_user_attr('acc', float(acc))
    trial.set_user_attr('nmi', float(nmi))
    trial.set_user_attr('ari', float(ari))
    return score


study = optuna.create_study(direction='maximize', study_name='dcg_lambda_tuning')
study.optimize(objective, n_trials=N_TRIALS)

print('Best score:', study.best_value)
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

safe_dataset = TUNE_DATASET.replace(' ', '_').replace('-', '_')
safe_mr = str(TUNE_MISSING_RATE).replace('.', 'p')
best_filename = f'best_lambda_{safe_dataset}_mr{safe_mr}.json'
best_path = os.path.join(DCG_DIR, best_filename)
legacy_best_path = os.path.join(DCG_DIR, 'best_lambda_params.json')

payload = {
    'dataset': TUNE_DATASET,
    'missing_rate': TUNE_MISSING_RATE,
    'tune_epochs': TUNE_EPOCHS,
    'n_trials': N_TRIALS,
    'best_score': study.best_value,
    'best_params': study.best_params,
    'best_metrics': {
        'acc': study.best_trial.user_attrs.get('acc'),
        'nmi': study.best_trial.user_attrs.get('nmi'),
        'ari': study.best_trial.user_attrs.get('ari'),
    },
}

# Primary per-dataset file
with open(best_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)

# Backward-compat convenience file
with open(legacy_best_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)

# Append summary row
summary_path = os.path.join(DCG_DIR, 'lambda_tuning_summary.csv')
row = {
    'timestamp': datetime.utcnow().isoformat(),
    'dataset': TUNE_DATASET,
    'missing_rate': TUNE_MISSING_RATE,
    'tune_epochs': TUNE_EPOCHS,
    'n_trials': N_TRIALS,
    'best_score': study.best_value,
    'acc': payload['best_metrics']['acc'],
    'nmi': payload['best_metrics']['nmi'],
    'ari': payload['best_metrics']['ari'],
    'best_filename': best_filename,
}
for k, v in study.best_params.items():
    row[k] = v

if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
else:
    df = pd.DataFrame([row])

df.to_csv(summary_path, index=False)

print('Saved dataset-specific params:', best_path)
print('Updated legacy params file:', legacy_best_path)
print('Updated tuning summary:', summary_path)

### Use tuned lambdas in final training

After tuning, load `best_lambda_params.json` and apply the best values in your training config (or pass them into `run.py` once we expose lambda CLI flags).

In [ ]:
import json

# Must match your intended training target
TARGET_DATASET = 'Synthetic3d'
TARGET_MISSING_RATE = 0.3

safe_dataset = TARGET_DATASET.replace(' ', '_').replace('-', '_')
safe_mr = str(TARGET_MISSING_RATE).replace('.', 'p')
best_filename = f'best_lambda_{safe_dataset}_mr{safe_mr}.json'
best_path = os.path.join(DCG_DIR, best_filename)

with open(best_path, 'r', encoding='utf-8') as f:
    best = json.load(f)

assert best['dataset'] == TARGET_DATASET, f"Dataset mismatch: expected {TARGET_DATASET}, found {best['dataset']}"
assert float(best['missing_rate']) == float(TARGET_MISSING_RATE), (
    f"Missing-rate mismatch: expected {TARGET_MISSING_RATE}, found {best['missing_rate']}"
)

print('Best tuned params loaded from:', best_path)
for k, v in best['best_params'].items():
    print(f'{k}: {v}')

## Final Training

Run this after the tuning cells if you want to use tuned lambda values.

In [ ]:
# Training runner
# Dataset IDs in run.py:
# 1: LandUse_21, 2: CUB, 3: HandWritten, 4: Multi-Fashion, 5: Synthetic3d

DATASET_ID = 5      # change as needed
EPOCHS = 200        # change as needed
TEST_TIME = 1
GPU_DEVICE = '0'
TARGET_MISSING_RATE = 0.3

DATASET_ID_TO_NAME = {
    1: 'LandUse_21',
    2: 'CUB',
    3: 'HandWritten',
    4: 'Multi-Fashion',
    5: 'Synthetic3d',
}

dataset_name = DATASET_ID_TO_NAME[DATASET_ID]
safe_dataset = dataset_name.replace(' ', '_').replace('-', '_')
safe_mr = str(TARGET_MISSING_RATE).replace('.', 'p')
LAMBDA_CONFIG = f'best_lambda_{safe_dataset}_mr{safe_mr}.json'

%cd {DCG_DIR}

if os.path.exists(LAMBDA_CONFIG):
    print('Using tuned params:', LAMBDA_CONFIG)
    !python run.py --dataset {DATASET_ID} --epoch {EPOCHS} --test_time {TEST_TIME} --devices {GPU_DEVICE} --lambda_config {LAMBDA_CONFIG}
else:
    print('No tuned file found, training without lambda_config:', LAMBDA_CONFIG)
    !python run.py --dataset {DATASET_ID} --epoch {EPOCHS} --test_time {TEST_TIME} --devices {GPU_DEVICE}